In [ ]:
import os
import random
from graphviz import Digraph
from IPython.display import Image, display


class Node:
    def __init__(self, key):
        self.key = key
        self.color = 0
        self.p = None
        self.left = None
        self.right = None


RED = 0
BLACK = 1
NIL = Node(None)
NIL.color = BLACK


class RBTree:
    def __init__(self):
        self.root = NIL

    def right_rotate(self, y):
        """右旋操作"""
        x = y.left
        beta = x.right
        pNode = y.p

        # 连接下级结点beta
        y.left = beta
        if beta != NIL:
            beta.p = y

        # 连接上级结点pNode
        x.p = pNode
        if pNode == NIL:
            self.root = x
        elif pNode.left == y:
            pNode.left = x
        else:
            pNode.right = x

        # 连接结点x和y
        x.right = y
        y.p = x

    def left_rotate(self, x):
        """左旋操作"""
        y = x.right
        beta = y.left
        pNode = x.p

        # 连接下级结点beta
        x.right = beta
        if beta != NIL:
            beta.p = x

        # 连接上级结点pNode
        y.p = pNode
        if pNode == NIL:
            self.root = y
        elif pNode.left == x:
            pNode.left = y
        else:
            pNode.right = y

        # 连接结点x和y
        y.left = x
        x.p = y

    def search(self, key):
        """搜索带有键的结点"""
        node = self.root
        while node != NIL:
            if key < node.key:
                node = node.left
            elif key > node.key:
                node = node.right
            else:
                return node
        return NIL

    def insert(self, key):
        """插入带有键的结点"""
        pNode = NIL
        node = self.root

        # 搜索直到匹配结点, 或者空结点
        while node != NIL:
            pNode = node
            if key < node.key:
                node = node.left
            elif key > node.key:
                node = node.right
            else:
                return

        # 建立当前结点的向上连接
        iNode = Node(key)
        iNode.p = pNode

        # 建立父结点的向下连接
        if pNode == NIL:
            self.root = iNode
        elif key < pNode.key:
            pNode.left = iNode
        else:
            pNode.right = iNode

        # 初始化为末尾红色结点
        iNode.left = NIL
        iNode.right = NIL
        iNode.color = RED

        self.insert_fixup(iNode)

    def insert_fixup(self, node):
        """插入调整"""
        # 此时node为红色, 如果父也是红色
        while node.p.color == RED:
            # 如果父是祖父的左结点
            if node.p == node.p.p.left:
                uncle = node.p.p.right

                # 如果叔结点为红色
                # 叔与父变黑, 祖父变红 且继续处理
                if uncle.color == RED:
                    node.p.color = BLACK
                    uncle.color = BLACK
                    node.p.p.color = RED
                    node = node.p.p

                # 如果叔结点为黑色
                # 旋转, 使红点分居两侧
                else:
                    if node == node.p.right:
                        node = node.p
                        self.left_rotate(node)
                    node.p.color = BLACK
                    node.p.p.color = RED
                    self.right_rotate(node.p.p)

            # 如果父是祖父的右结点
            else:
                uncle = node.p.p.left

                # 如果叔结点为红色
                # 叔与父变黑, 祖父变红 且继续处理
                if uncle.color == RED:
                    node.p.color = BLACK
                    uncle.color = BLACK
                    node.p.p.color = RED
                    node = node.p.p

                # 如果叔结点为黑色
                # 旋转, 使红点分居两侧
                else:
                    if node == node.p.left:
                        node = node.p
                        self.right_rotate(node)
                    node.p.color = BLACK
                    node.p.p.color = RED
                    self.left_rotate(node.p.p)

        # 最后将根结点变黑
        self.root.color = BLACK

    def _transplant(self, old, new):
        """移动替换整棵子树"""
        # 更新当前结点的向上连接
        new.p = old.p

        # 更新父结点的向下连接
        if old.p == NIL:
            self.root = new
        elif old == old.p.left:
            old.p.left = new
        else:
            old.p.right = new

    def _minimum(self, node):
        """返回子树的最小结点"""
        while node.left != NIL:
            node = node.left
        return node

    def delete(self, key):
        node = self.search(key)

        y = node
        y_original_color = y.color

        if node.left == NIL:
            x = node.right
            self._transplant(node, node.right)

        elif node.right == NIL:
            x = node.left
            self._transplant(node, node.left)

        else:
            y = self._minimum(node.right)
            y_original_color = y.color
            x = y.right
            if y.p == node:
                x.p = y
            else:
                self._transplant(y, y.right)
                y.right = node.right
                y.right.p = y
            self._transplant(node, y)
            y.left = node.left
            y.left.p = y
            y.color = node.color

        if y_original_color == BLACK:
            self.delete_fixup(x)

    def delete_fixup(self, node):
        while node != self.root and node.color == BLACK:

            if node == node.p.left:
                w = node.p.right
                if w.color == RED:
                    w.color = BLACK
                    node.p.color = RED
                    self.left_rotate(node.p)
                    w = node.p.right
                if w.left.color == BLACK and w.right.color == BLACK:
                    w.color = RED
                    node = node.p
                else:
                    if w.right.color == BLACK:
                        w.left.color = BLACK
                        w.color = RED
                        self.right_rotate(w)
                        w = node.p.right
                    w.color = node.p.color
                    node.p.color = BLACK
                    w.right.color = BLACK
                    self.left_rotate(node.p)
                    node = self.root

            else:
                w = node.p.left
                if w.color == RED:
                    w.color = BLACK
                    node.p.color = RED
                    self.right_rotate(node.p)
                    w = node.p.left
                if w.right.color == BLACK and w.left.color == BLACK:
                    w.color = RED
                    node = node.p
                else:
                    if w.left.color == BLACK:
                        w.right.color = BLACK
                        w.color = RED
                        self.left_rotate(w)
                        w = node.p.left
                    w.color = node.p.color
                    node.p.color = BLACK
                    w.left.color = BLACK
                    self.right_rotate(node.p)
                    node = self.root

        node.color = BLACK

    def draw_tree(self, dot=None, node=None):
        """绘制红黑树"""
        if dot is None:
            # 初始化图
            dot = Digraph()
            dot.attr("node", shape="circle")
            dot.attr("edge", dir="none")

            # 递归添加每层的结点和边
            self.draw_tree(dot, self.root)

            # 渲染并显示图像
            dot.render("bst", format="png", cleanup=True)
            display(Image("bst.png"))
            os.remove("bst.png")
        else:
            if node != NIL:
                str_node = str(node.key)
                color = "lightcoral" if node.color == RED else ""
                dot.node(str_node, str_node, style="filled", fillcolor=color)
                if node.left != NIL:
                    str_left = str(node.left.key)
                    dot.edge(str_node, str_left, constraint="true")
                    self.draw_tree(dot, node.left)
                else:
                    dot.node(str_node + "L", "", shape="point")
                    dot.edge(str_node, str_node + "L", constraint="true")
                if node.right != NIL:
                    str_right = str(node.right.key)
                    dot.edge(str_node, str_right, constraint="true")
                    self.draw_tree(dot, node.right)
                else:
                    dot.node(str_node + "R", "", shape="point")
                    dot.edge(str_node, str_node + "R", constraint="true")